In [3]:
import joblib
import pandas as pd
from transformers import pipeline
import os

In [4]:
print("Loading the Comparison Engine (SVM) and Writer (LLM)...")
best_model = joblib.load('model_svm_tuned.pkl')
tfidf = joblib.load('tfidf_vectorizer.pkl')

Loading the Comparison Engine (SVM) and Writer (LLM)...


In [5]:
generator = pipeline("text-generation", model="gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [6]:
if os.path.exists('final_checked_requirements.csv'):
    df = pd.read_csv('final_checked_requirements.csv')
else:
    # Fallback if the previous file isn't found
    df = pd.read_csv('validated_requirements.csv')

def ai_generate(prompt_text, max_words=20):
    """Internal function to handle LLM generation"""
    results = generator(
        prompt_text, 
        max_new_tokens=max_words, 
        num_return_sequences=1, 
        pad_token_id=50256,
        truncation=True
    )
    # Extract only the generated part and clean it
    output = results[0]['generated_text'].replace(prompt_text, "").strip()
    # Stop at the first period to keep it as a single requirement
    return output.split('.')[0] + "." if "." in output else output

In [7]:
print("\n[Action] Completing incomplete requirements...")
for idx, row in df.iterrows():
    if row['completeness_status'] != "Complete":
        original = row['requirement_sentence']
        # We give the AI a 'prompt' to set the context
        prompt = f"Software Requirement: {original}"
        completion = ai_generate(prompt)
        
        df.at[idx, 'requirement_sentence'] = f"{original} {completion}"
        df.at[idx, 'completeness_status'] = "Completed by AI"
        print(f"Fixed: {original} -> {df.at[idx, 'requirement_sentence']}")


[Action] Completing incomplete requirements...


In [8]:
print("\n[Action] Comparing against training patterns...")
missing_categories = []
if df['Security'].sum() == 0: missing_categories.append("Security")
if df['Reliability'].sum() == 0: missing_categories.append("Reliability")


[Action] Comparing against training patterns...


In [9]:
new_reqs = []
for category in missing_categories:
    print(f"Adding a missing {category} requirement based on dataset patterns...")
    # Prompt the AI to write a requirement specifically for that category
    prompt = f"Requirement for {category}: The system shall"
    new_text = "The system shall " + ai_generate(prompt)
    
    new_data = {
        'requirement_sentence': new_text,
        'NFR_boolean': 1,
        'Security': 1 if category == "Security" else 0,
        'Reliability': 1 if category == "Reliability" else 0,
        'completeness_status': 'AI Suggested (Missing Category)'
    }
    new_reqs.append(new_data)

Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=20) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Adding a missing Reliability requirement based on dataset patterns...


In [10]:
if new_reqs:
    df = pd.concat([df, pd.DataFrame(new_reqs)], ignore_index=True)

In [11]:
df.to_csv('final_expanded_requirements.csv', index=False)
print("\n--- Final Requirement Set Ready ---")
display(df[['requirement_sentence', 'completeness_status']].tail(5))


--- Final Requirement Set Ready ---


,requirement_sentence,completeness_status
6,The system shall provide the ability to genera...,Complete
7,The system shall provide the ability to export...,Complete
8,This export on hardcopy and electronic output ...,Complete
9,The system shall provide the ability to create...,Complete
10,The system shall permit the use of certain com...,AI Suggested (Missing Category)
